In [121]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, false_discovery_control
from collections import Counter

In [91]:
df = pd.read_csv("../data/interviews/final_prototype_interviews.csv")
quantitative_df = df[['Question','Respondent 1','Respondent 2']].iloc[0:24].T
quantitative_df

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
Question,Observation: Count how many times the user too...,"Q1. ""Completing the reporting process in this ...","Q2. ""Using this app was an enjoyable experienc...","Q3. ""The app gave me clear information about w...","Q4. ""If I submitted a report using this app, I...","Q5. ""If this app were available in my city, I ...",Observation: Count how many times the user too...,"Q1. ""Completing the reporting process in this ...","Q2. ""Using this app was an enjoyable experienc...","Q3. ""The app gave me clear information about w...",...,"Q2. ""Using this app was an enjoyable experienc...","Q3. ""The app gave me clear information about w...","Q4. ""If I submitted a report using this app, I...","Q5. ""If this app were available in my city, I ...",Observation: Count how many times the user too...,"Q1. ""Completing the reporting process in this ...","Q2. ""Using this app was an enjoyable experienc...","Q3. ""The app gave me clear information about w...","Q4. ""If I submitted a report using this app, I...","Q5. ""If this app were available in my city, I ..."
Respondent 1,1,5,5,4,4,5,0,5,4,5,...,5,5,4,4,0,5,4,3,4,5
Respondent 2,1,5,4,4,5,5,0,4,4,5,...,5,5,5,5,0,5,4,5,5,5


In [92]:
# Step 1: Promote the first row (questions) as column headers
quantitative_df.columns = quantitative_df.iloc[0]        # set Question row as headers
quantitative_df = quantitative_df.drop(quantitative_df.index[0])  # drop the Question row
# Now index is: ['Respondent 1', 'Respondent 2']

# Step 2: Deduplicate column names (Q1, Q2... repeat across interviewers)
counts = Counter()
new_cols = []
for col in quantitative_df.columns:
    counts[col] += 1
    new_cols.append(f"{col} [{counts[col]}]" if counts[col] > 1 else col)
quantitative_df.columns = new_cols

# Step 3: Stack into long format, then restore question name (strip suffix)
df_long = (
    quantitative_df
    .reset_index()
    .rename(columns={'index': 'Respondent'})
    .melt(id_vars='Respondent', var_name='Question', value_name='Answer')
    .assign(Question=lambda x: x['Question'].str.replace(r'\s\[\d+\]$', '', regex=True))
    .sort_values(['Question', 'Respondent'])
    .reset_index(drop=True)
)

In [93]:
df_long['Answer'] = pd.to_numeric(df_long['Answer'], errors='coerce')

In [95]:
df_stats = (
    df_long
    .groupby('Question')['Answer']
    .describe()
    .round(2)
)

In [97]:
# baseline scores of original prototype A
column_map = {'Q2' : 'A_easy','Q3' : 'A_enjoyable','Q4' : 'A_status','Q5' : 'A_action',
              'Q7' : 'B_easy','Q8' : 'B_enjoyable','Q9' : 'B_status','Q10' : 'B_action',
              'Q12' : 'C_easy','Q13' : 'C_enjoyable','Q14' : 'C_status','Q15' : 'C_action',
              'Q16' : 'most_preferred','Q17' : 'least_preferred',
              'Q18' : 'MP_why','Q19' : 'LP_why'}
results = pd.read_csv("../data/surveys/survey_results.csv")
results.dropna(axis=1, how='all', inplace=True)
results.set_index('response', inplace=True)
results.rename(columns=column_map, inplace=True)
# Drop bogus responses
results.drop(index=results.index[[3, 4, 8, 22]], inplace=True)
baseline = results[['A_easy', 'A_enjoyable', 'A_status', 'A_action']]

# get baseline scores as arrays
q1_easy_baseline = baseline['A_easy'].values
q2_enjoyable_baseline = baseline['A_enjoyable'].values
q3_status_baseline = baseline['A_status'].values
q4_action_baseline = baseline['A_action'].values


In [129]:
print(f"Q1 Baseline Mean: {q1_easy_baseline.mean()}")
print(f"Q1 Baseline Median: {np.median(q1_easy_baseline)}")

print(f"Q2 Baseline Mean: {q2_enjoyable_baseline.mean()}")
print(f"Q2 Baseline Median: {np.median(q2_enjoyable_baseline)}")

print(f"Q3 Baseline Mean: {q3_status_baseline.mean()}")
print(f"Q3 Baseline Median: {np.median(q3_status_baseline)}")

print(f"Q4 Baseline Mean: {q4_action_baseline.mean()}")
print(f"Q4 Baseline Median: {np.median(q4_action_baseline)}")


Q1 Baseline Mean: 4.368421052631579
Q1 Baseline Median: 4.0
Q2 Baseline Mean: 4.052631578947368
Q2 Baseline Median: 4.0
Q3 Baseline Mean: 4.157894736842105
Q3 Baseline Median: 4.0
Q4 Baseline Mean: 3.9473684210526314
Q4 Baseline Median: 4.0


In [113]:
# get new scores as arrays
new_subset = df_long[['Question','Answer']].iloc[8:40]

q1_easy_new = new_subset['Answer'].iloc[0:8].values
q2_enjoyable_new = new_subset['Answer'].iloc[8:16].values
q3_status_new = new_subset['Answer'].iloc[16:24].values
q4_action_new = new_subset['Answer'].iloc[24:32].values

In [130]:
print(f"Q1 New Mean: {q1_easy_new.mean()}")
print(f"Q1 New Median: {np.median(q1_easy_new)}")

print(f"Q2 New Mean: {q2_enjoyable_new.mean()}")
print(f"Q2 New Median: {np.median(q2_enjoyable_new)}")

print(f"Q3 New Mean: {q3_status_new.mean()}")
print(f"Q3 New Median: {np.median(q3_status_new)}")

print(f"Q4 New Mean: {q4_action_new.mean()}")
print(f"Q4 New Median: {np.median(q4_action_new)}")

Q1 New Mean: 4.875
Q1 New Median: 5.0
Q2 New Mean: 4.375
Q2 New Median: 4.0
Q3 New Mean: 4.5
Q3 New Median: 5.0
Q4 New Mean: 4.5
Q4 New Median: 4.5


In [135]:
# Mann Whitney Values
alpha = .05
# U stat - from Mann Whitney Table
u_stat = 38

print(f"U stat: {u_stat}\nAlpha: {alpha}")

U stat: 38
Alpha: 0.05


In [114]:
q1_stat, q1_p = mannwhitneyu(q1_easy_baseline, q1_easy_new, method='auto')
q1_stat, q1_p

(np.float64(37.5), np.float64(0.019809878085184373))

In [115]:
q2_stat, q2_p = mannwhitneyu(q2_enjoyable_baseline, q2_enjoyable_new, method='auto')
q2_stat, q2_p

(np.float64(57.5), np.float64(0.285357818133414))

In [116]:
q3_stat, q3_p = mannwhitneyu(q3_status_baseline, q3_status_new, method='auto')
q3_stat, q3_p

(np.float64(52.5), np.float64(0.17465911984750548))

In [118]:
q4_stat, q4_p = mannwhitneyu(q4_action_baseline, q4_action_new, method='auto')
q4_stat, q4_p

(np.float64(56.0), np.float64(0.26534384257650623))

In [136]:
# False Discovery Control with Benjamini Hochberg
ps_adjusted = false_discovery_control(ps=[q1_p, q2_p, q3_p, q4_p],method='bh')
ps_adjusted

array([0.07923951, 0.28535782, 0.28535782, 0.28535782])

In [137]:
# Effect Size, Perception of Ease (Q1)
U_baseline = 37.5
U_new = (n1 * n2) - U_baseline

f = U_new / (n1 * n2)

f

0.7532894736842105